# 🏗️ Notebook: Structured Outputs with Pydantic

LLMs return free text. For most applications we need data we can work with in code, e.g. a dict with fixed fields. In this notebook you'll learn how to get exactly that:

1. **Pydantic basics:** describe the shape of your data as a Python class
2. **Structured outputs:** force the LLM to answer in exactly that shape

## 📚 Sources

- [Pydantic Documentation](https://docs.pydantic.dev/)
- [OpenAI: Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs?api-mode=chat)
- [Ollama Structured Outputs](https://ollama.com/blog/structured-outputs)

# Part 1: Pydantic Basics

## 1. A Basic Model

A Pydantic model is a class that inherits from `BaseModel`. Instead of writing `__init__`, you simply list the attributes with their types. Pydantic generates the constructor and validation for you.

In [23]:
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int

person = Person(name="Alice", age=30)
print(person)
print(person.name, person.age)

name='Alice' age=30
Alice 30


## 2. Validation

Compatible values are converted automatically (`"25"` → `25`). Values that don't fit raise a `ValidationError`. This is the whole point: bad data is caught instead of silently accepted.

In [24]:
from pydantic import ValidationError

print(Person(name="Bob", age="25"))  # "25" is converted to int

try:
    Person(name="Charlie", age="not a number")
except ValidationError as e:
    print(e)

name='Bob' age=25
1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not a number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


## 3. `Field()`, `Literal` and Optional Fields

- `Field()` adds constraints, e.g. `gt`/`ge`/`lt`/`le` (greater/less than (or equal)) or `min_length`/`max_length`.
- `Literal[...]` allows only a fixed set of values, perfect for categories.
- `X | None = None` makes a field optional (default `None`).

In [25]:
from typing import Literal
from pydantic import Field

class Product(BaseModel):
    name: str
    price: float = Field(gt=0)                       # must be > 0
    category: Literal["food", "drinks", "other"]     # only these three values
    note: str | None = None                          # optional

print(Product(name="Coffee", price=2.5, category="drinks"))

try:
    Product(name="Coffee", price=-1, category="clothes")
except ValidationError as e:
    print(e)

name='Coffee' price=2.5 category='drinks' note=None
2 validation errors for Product
price
  Input should be greater than 0 [type=greater_than, input_value=-1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
category
  Input should be 'food', 'drinks' or 'other' [type=literal_error, input_value='clothes', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


## 4. Nested Models

Models can contain other models, also as lists. Pydantic validates the whole structure.

In [26]:
class Address(BaseModel):
    city: str
    zip_code: str

class Customer(BaseModel):
    name: str
    addresses: list[Address]

customer = Customer(name="Frank", addresses=[{"city": "Munich", "zip_code": "80331"}])
print(customer.addresses[0].city)

Munich


## 5. JSON In, JSON Out

Two methods connect Pydantic to LLMs:

- `model_validate_json()` parses a JSON string into a validated object. Remember: an LLM only ever generates text. Even when it "returns JSON", it is just a string that looks like JSON. With this method we turn that string into a real Python object.
- `model_json_schema()` turns the model into a **JSON Schema**, a standard description of the expected structure. This schema is what we send to the LLM.

In [27]:
import json

person = Person.model_validate_json('{"name": "Grace", "age": 28}')
print(person)
print(person.model_dump())  # back to a dict

print(json.dumps(Person.model_json_schema(), indent=2))

name='Grace' age=28
{'name': 'Grace', 'age': 28}
{
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "age": {
      "title": "Age",
      "type": "integer"
    }
  },
  "required": [
    "name",
    "age"
  ],
  "title": "Person",
  "type": "object"
}


# Part 2: Structured Outputs with LLMs

**Structured outputs** restrict the model's output to a given JSON schema. We pass a Pydantic model as `response_format` and the server (here Ollama via its OpenAI-compatible API) makes sure the answer matches the schema.

In [28]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads LLM_HOST, LLM_PORT and LLM_API_KEY from the .env file (see setup.md)

LLM_HOST = os.environ["LLM_HOST"]
LLM_PORT = os.environ["LLM_PORT"]
LLM_API_KEY = os.environ["LLM_API_KEY"]
LLM_URL = f"http://{LLM_HOST}:{LLM_PORT}/v1"
LLM_REASONING = "gemma4:26b"

client = OpenAI(base_url=LLM_URL, api_key=LLM_API_KEY)

Two habits for the examples below:

- **Include the schema in the prompt.** `response_format` only enforces the *shape*. The model still needs to know what each field *means*, so we also put the schema into the prompt.
- **Disable thinking for simple tasks** with `reasoning_effort="none"`. It saves time without hurting quality here.

## Example 1: Classification

`SentimentResult.model_json_schema()` turns the Pydantic class into a **JSON Schema**: a Python dict that describes every field with its name, type and allowed values (here: `sentiment` is a string, one of `"positive"`, `"neutral"`, `"negative"`). `json.dumps(..., indent=2)` turns this dict into a readable string, so we can put it into the prompt. That way the model can actually read which fields it should fill and what they mean.

In [29]:
class SentimentResult(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]

prompt = f"""Classify the sentiment of this text: "I love programming in Python!"

Answer as JSON matching this schema:
{json.dumps(SentimentResult.model_json_schema(), indent=2)}
"""

completion = client.chat.completions.parse(
    model=LLM_REASONING,
    temperature=0,
    reasoning_effort="none",
    messages=[{"role": "user", "content": prompt}],
    response_format=SentimentResult,
)

result = completion.choices[0].message.parsed  # already a SentimentResult object
print(result)
print(result.sentiment)

sentiment='positive'
positive


`.message.content` contains the raw JSON string, `.message.parsed` the validated Pydantic object.

## Example 2: Extracting Information from Text

The same pattern works for nested structures, e.g. a list of courses. Note that `semester` is optional: the second course has no semester in the text.

In [30]:
class Course(BaseModel):
    title: str
    professor: str
    credits: int
    semester: str | None = None

class CourseList(BaseModel):
    courses: list[Course]

text = """I'm enrolled in two courses. "Introduction to Machine Learning" is taught by Prof. Müller,
worth 6 credits and offered in the winter semester. I'm also taking "Database Systems",
a 5-credit course taught by Prof. Schmidt."""

prompt = f"""Extract all courses from this text:
{text}

Answer as JSON matching this schema:
{json.dumps(CourseList.model_json_schema(), indent=2)}
"""

completion = client.chat.completions.parse(
    model=LLM_REASONING,
    temperature=0,
    reasoning_effort="none",
    messages=[{"role": "user", "content": prompt}],
    response_format=CourseList,
)

for course in completion.choices[0].message.parsed.courses:
    print(course)

title='Introduction to Machine Learning' professor='Prof. Müller' credits=6 semester='winter semester'
title='Database Systems' professor='Prof. Schmidt' credits=5 semester=None


### Exercise 1

Let the LLM generate dummy data for **2 researchers**.

| Field          | Description                                                      |
| -------------- | ---------------------------------------------------------------- |
| `name`         | Name of the researcher                                           |
| `age`          | Age between 25 and 80                                            |
| `field`        | One of: Computer Science, Biology, Physics, Mathematics          |
| `publications` | List of publications, each with `title` and `year` (1900–2025)   |

💡 Use `Field()` and `Literal`, and include the JSON schema in the prompt.

<details>
<summary><b>Show Solution</b></summary>

```python
class Publication(BaseModel):
    title: str
    year: int = Field(ge=1900, le=2025)

class Researcher(BaseModel):
    name: str
    age: int = Field(ge=25, le=80)
    field: Literal["Computer Science", "Biology", "Physics", "Mathematics"]
    publications: list[Publication]

class ResearcherList(BaseModel):
    researchers: list[Researcher] = Field(min_length=2, max_length=2)

prompt = f"""Generate 2 fictional researchers.

Answer as JSON matching this schema:
{json.dumps(ResearcherList.model_json_schema(), indent=2)}
"""

completion = client.chat.completions.parse(
    model=LLM_REASONING,
    temperature=0,
    reasoning_effort="none",
    messages=[{"role": "user", "content": prompt}],
    response_format=ResearcherList,
)

for researcher in completion.choices[0].message.parsed.researchers:
    print(researcher)
```

</details>

In [31]:
# Your code goes here...

## ADVANCED – Example 3: Regex Instead of JSON

*Optional: this goes beyond the standard JSON workflow and is specific to Ollama.*

Ollama's native API (`/api/chat`, not the OpenAI-compatible `/v1`) can also constrain the output to a **regex pattern**. The `openai` package silently ignores this, so we use `requests` here.

⚠️ Keep patterns tight, e.g. `[A-Z][a-z]+` or `[0-9]{1,3}`. An open `.*` can make Ollama hang for minutes, and loose patterns like `[A-Za-z ]+` let the model add stray spaces.

Since the format is technically a JSON schema of type `string`, Ollama returns a JSON string **including the quotes** (`"Name: ..."`). `json.loads()` turns it into a normal Python string.

In [32]:
import requests

data = {
    "model": LLM_REASONING,
    "messages": [{"role": "user", "content": "Create a name and age for a random person."}],
    "stream": False,
    "think": False,
    "format": {"type": "string", "pattern": "^Name: [A-Z][a-z]+ [A-Z][a-z]+, Age: [0-9]{1,3}$"},
}

response = requests.post(
    f"http://{LLM_HOST}:{LLM_PORT}/api/chat",
    json=data,
    headers={"Authorization": f"Bearer {LLM_API_KEY}"},
)
raw = response.json()["message"]["content"]
print(raw)              # JSON string with quotes
print(json.loads(raw))  # plain Python string

"Name: Elena Vance, Age: 34"
Name: Elena Vance, Age: 34


### Exercise 2

Extract a structured order from this customer email:

> Hi, I'd like to order 3 notebooks at 2.50 EUR each and 1 backpack for 39.90 EUR. Please ship it to Regensburg, 93053. Thanks, Lena Weber

The result should contain the customer name, the shipping address (city, zip code) and a list of items (name, quantity, unit price).

<details>
<summary><b>Show Solution</b></summary>

```python
class Item(BaseModel):
    name: str
    quantity: int = Field(ge=1)
    unit_price_eur: float = Field(gt=0)

class Order(BaseModel):
    customer_name: str
    shipping_address: Address  # reuses the Address model from Part 1
    items: list[Item]

email = """Hi, I'd like to order 3 notebooks at 2.50 EUR each and 1 backpack for 39.90 EUR.
Please ship it to Regensburg, 93053. Thanks, Lena Weber"""

prompt = f"""Extract the order from this email:
{email}

Answer as JSON matching this schema:
{json.dumps(Order.model_json_schema(), indent=2)}
"""

completion = client.chat.completions.parse(
    model=LLM_REASONING,
    temperature=0,
    reasoning_effort="none",
    messages=[{"role": "user", "content": prompt}],
    response_format=Order,
)

order = completion.choices[0].message.parsed
print(order)
print("Total:", sum(item.quantity * item.unit_price_eur for item in order.items))
```

</details>

In [33]:
# Your code goes here...